## Sudoku Solver Using Constraint Satisfaction Problems (CSP)


**CSP formulation**
- **Variables:** Each unfilled cell `(row, column)`.
- **Domains:** Legal digits from 1 through 9 for each unfilled cell.
- **Constraints:** No repeated digit in a row, column, or 3 × 3 box.
- **Search:** MRV chooses the next cell; LCV chooses which value to try first; forward checking eliminates that value from unassigned peers' domains. If a domain becomes empty, the branch fails and backtracking tries another value.



In [1]:
# Input Sudoku: 0 represents an empty cell.
puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

In [2]:
DIGITS = set(range(1, 10))
CELLS = [(row, col) for row in range(9) for col in range(9)]


def display_grid(grid):
    """Display a 9×9 grid with 3×3 subgrid separators."""
    for row in range(9):
        if row in (3, 6):
            print("------+-------+------")
        parts = []
        for col in range(9):
            value = grid[row][col]
            parts.append(str(value) if value else ".")
            if col in (2, 5):
                parts.append("|")
        print(" ".join(parts))


def get_peers(cell):
    """Return all other cells in the same row, column, or 3×3 subgrid."""
    row, col = cell
    box_row, box_col = (row // 3) * 3, (col // 3) * 3
    peers = {(row, c) for c in range(9)}
    peers.update((r, col) for r in range(9))
    peers.update((r, c) for r in range(box_row, box_row + 3)
                          for c in range(box_col, box_col + 3))
    peers.discard(cell)
    return peers


PEERS = {cell: get_peers(cell) for cell in CELLS}


def is_valid_initial_grid(grid):
    """Verify shape, allowed digits, and consistency of fixed entries."""
    if len(grid) != 9 or any(len(row) != 9 for row in grid):
        return False

    for row in grid:
        if any(type(value) is not int or value not in range(10) for value in row):
            return False

    for row, col in CELLS:
        value = grid[row][col]
        if value != 0 and any(grid[r][c] == value for r, c in PEERS[(row, col)]):
            return False
    return True


def is_solved(grid):
    """Check that the board is completely filled and respects all constraints."""
    return is_valid_initial_grid(grid) and all(grid[r][c] != 0 for r, c in CELLS)


### 2. CSP domains, MRV, LCV, and forward checking

These functions implement each required CSP technique separately. Domains are copied at each recursive branch so the solver can safely restore them during backtracking.

In [3]:
def initialize_domains(grid):
    """Create a set of legal digits for every empty cell; return None on failure."""
    domains = {}
    for row, col in CELLS:
        if grid[row][col] == 0:
            used = {grid[r][c] for r, c in PEERS[(row, col)] if grid[r][c] != 0}
            possible = DIGITS - used
            if not possible:
                return None
            domains[(row, col)] = possible
    return domains


def select_mrv_cell(domains):
    """MRV: choose the unassigned cell with the smallest remaining domain."""
    return min(domains, key=lambda cell: len(domains[cell]))


def order_lcv_values(cell, domains):
    """LCV: prefer the digit that rules out the fewest peer-domain values."""
    def eliminated_options(value):
        return sum(value in domains[peer]
                   for peer in PEERS[cell] if peer in domains)

    return sorted(domains[cell], key=lambda value: (eliminated_options(value), value))


def forward_check(domains, cell, value):
    """Remove value from unassigned peers; return None if a domain empties."""
    updated_domains = {variable: options.copy()
                       for variable, options in domains.items() if variable != cell}

    for peer in PEERS[cell]:
        if peer in updated_domains:
            updated_domains[peer].discard(value)
            if not updated_domains[peer]:
                return None

    return updated_domains


### 3. Recursive backtracking solver
Assign a value chosen by LCV, propagate the assignment using forward checking, then recurse. On a failed branch, restore the empty cell and try the next value.

In [4]:
def backtrack(grid, domains, stats):
    """Solve recursively with MRV + LCV + forward checking."""
    stats["recursive_calls"] += 1

    if not domains:
        return True

    cell = select_mrv_cell(domains)
    row, col = cell

    for value in order_lcv_values(cell, domains):
        grid[row][col] = value
        stats["assignments_tried"] += 1

        remaining_domains = forward_check(domains, cell, value)
        if remaining_domains is not None and backtrack(grid, remaining_domains, stats):
            return True

        grid[row][col] = 0  # Undo the choice before trying another value.
        stats["backtracks"] += 1

    return False


def solve_sudoku(puzzle):
    """Return (solved grid or None, search statistics) without changing input."""
    stats = {"recursive_calls": 0, "assignments_tried": 0, "backtracks": 0}

    if not is_valid_initial_grid(puzzle):
        return None, stats

    grid = [row.copy() for row in puzzle]
    domains = initialize_domains(grid)
    if domains is None:
        return None, stats

    if backtrack(grid, domains, stats):
        return grid, stats

    return None, stats


In [5]:
print("INITIAL SUDOKU PUZZLE")
display_grid(puzzle)

solution, statistics = solve_sudoku(puzzle)

print("\nSOLVED SUDOKU PUZZLE")
if solution is None:
    print("No valid solution exists for the given puzzle.")
else:
    display_grid(solution)
    print("\nAll Sudoku constraints satisfied:", is_solved(solution))

print("\nSearch statistics:", statistics)

INITIAL SUDOKU PUZZLE
5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
------+-------+------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
------+-------+------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9

SOLVED SUDOKU PUZZLE
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
------+-------+------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
------+-------+------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9

All Sudoku constraints satisfied: True

Search statistics: {'recursive_calls': 52, 'assignments_tried': 51, 'backtracks': 0}


### 5. Failure-detection test
A contradictory starting puzzle must be rejected rather than returning an incorrect grid. The solver also returns `None` when a consistent starting configuration leads to no solution.

In [6]:
# Make a contradictory puzzle by adding a second 5 in the first row.
invalid_puzzle = [row.copy() for row in puzzle]
invalid_puzzle[0][2] = 5

invalid_solution, invalid_stats = solve_sudoku(invalid_puzzle)
print("Invalid initial puzzle detected:", invalid_solution is None)

# This puzzle is initially consistent but cannot be completed.
unsolvable_puzzle = [row.copy() for row in puzzle]
unsolvable_puzzle[0][2] = 1  # Legal locally, but incompatible with a full solution.

print("Unsolvable puzzle passes initial consistency check:",
      is_valid_initial_grid(unsolvable_puzzle))
unsolvable_solution, unsolvable_stats = solve_sudoku(unsolvable_puzzle)
print("No-solution outcome detected:", unsolvable_solution is None)
print("Search statistics for unsolvable puzzle:", unsolvable_stats)

Invalid initial puzzle detected: True
Unsolvable puzzle passes initial consistency check: True
No-solution outcome detected: True
Search statistics for unsolvable puzzle: {'recursive_calls': 3, 'assignments_tried': 3, 'backtracks': 3}
